# Cricket Match Outcome Predictor — Model Explorer

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, roc_curve, precision_recall_curve

checkpoint = torch.load('../results/cricket_outcome_model.pt', map_location='cpu', weights_only=False)

print('Checkpoint loaded successfully')
print(f'Architecture : {checkpoint["input_dim"]} inputs  →  {checkpoint["hidden_dims"]}  →  2 outputs')
print(f'Dropout      : {checkpoint["dropout"]}')
print(f'\nTest set performance (600 held-out matches):')
for k, v in checkpoint['test_metrics'].items():
    print(f'  {k:18s}: {v:.4f}')

In [2]:
print('Input features  :', checkpoint['input_dim'])
print('Hidden layers   :', checkpoint['hidden_dims'])
print('Dropout         :', checkpoint['dropout'])
print()
print('Test set results:')
for k, v in checkpoint['test_metrics'].items():
    print(f'  {k:18s}: {v:.4f}')

Input features  : 29
Hidden layers   : [128, 64, 32]
Dropout         : 0.3

Test set results:
  auc               : 0.7469
  avg_precision     : 0.8686
  brier             : 0.1746


In [ ]:
feature_names = checkpoint['feature_names']
print(f'Model expects {len(feature_names)} features (in this exact order):\n')
for i, name in enumerate(feature_names):
    group = 'Engineered' if i >= 20 else 'Raw'
    print(f'  {i+1:2d}. {name:30s}  [{group}]')

In [4]:
import torch.nn as nn

class CricketOutcomeNet(nn.Module):
    def __init__(self, input_dim, hidden_dims=[128, 64, 32], dropout=0.3):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 2))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

model = CricketOutcomeNet(
    input_dim=checkpoint['input_dim'],
    hidden_dims=checkpoint['hidden_dims'],
    dropout=checkpoint['dropout']
)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print('Model loaded and ready')
print(model)

Model loaded and ready
CricketOutcomeNet(
  (network): Sequential(
    (0): Linear(in_features=29, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.3, inplace=False)
    (12): Linear(in_features=32, out_features=2, bias=True)
  )
)


In [ ]:
# Sample prediction — edit any value below to simulate a match scenario

feature_names = checkpoint['feature_names']
mean  = checkpoint['scaler_mean']
scale = checkpoint['scaler_scale']

sample_values = {
    # Pre-match features
    't1_form_win_pct'    : 0.70,   # Team 1 won 70% of last 10 games
    't2_form_win_pct'    : 0.50,
    'h2h_ratio'          : 0.60,   # T1 wins 60% of head-to-head matches
    't1_batting_avg'     : 32.0,
    't2_batting_avg'     : 28.0,
    't1_bowling_economy' : 7.8,
    't2_bowling_economy' : 8.5,
    't1_venue_win_pct'   : 0.55,
    'toss_won_by_t1'     : 1,      # 1 = T1 won toss, 0 = T2
    'toss_decision_bat'  : 1,      # 1 = chose to bat, 0 = field

    # Powerplay (overs 1-6)
    'pp_runs_t1'         : 52,
    'pp_wickets_t1'      : 1,

    # Mid innings (overs 7-15)
    'mid_runs_t1'        : 75,
    'mid_wickets_t1'     : 2,

    # End of T1 innings
    'wickets_in_hand_t1' : 8,
    'crr_at_over15'      : 8.2,

    # T2 chase context
    't2_pp_runs'         : 44,
    't2_pp_wickets'      : 2,
    't2_wickets_in_hand' : 7,
    'required_run_rate'  : 9.1,

    # Engineered features — derived from the raw values above
    'form_diff'           : 0.70 - 0.50,
    'batting_avg_diff'    : 32.0 - 28.0,
    'bowling_econ_diff'   : 8.5 - 7.8,
    'pp_run_rate_t1'      : round(52 / 6, 2),
    'pp_wicket_pressure'  : round(1 / 6, 3),
    'run_rate_vs_required': round(8.2 - 9.1, 2),
    'wicket_momentum'     : 8 - 7,
    'powerplay_advantage' : 52 - 44,
    'pitch_encoded'       : 1,     # 1=batting, 0=balanced, -1=bowling
}

row = np.array([[sample_values[f] for f in feature_names]], dtype=np.float32)
row_scaled = ((row - mean) / scale).astype(np.float32)

with torch.no_grad():
    probs = torch.softmax(model(torch.tensor(row_scaled)), dim=1)[0]

p1 = probs[1].item()
p2 = probs[0].item()
winner = 'Team 1' if p1 > 0.5 else 'Team 2'

print(f'P(Team 1 wins) : {p1:.1%}')
print(f'P(Team 2 wins) : {p2:.1%}')
print(f'\nPrediction : {winner} wins  (confidence: {max(p1, p2):.1%})')

In [ ]:
# Full dataset predictions — uses the feature engineering pipeline so engineered features are correct
from src.features import get_feature_matrix, get_scaler

df_raw = pd.read_csv('../data/matches.csv')
X, y, feat_names = get_feature_matrix(df_raw)

mean  = checkpoint['scaler_mean']
scale = checkpoint['scaler_scale']

X_scaled = ((X - mean) / scale).astype(np.float32)

with torch.no_grad():
    all_probs = torch.softmax(model(torch.tensor(X_scaled)), dim=1)[:, 1].numpy()

preds = (all_probs >= 0.5).astype(int)
accuracy = (preds == y).mean()

print(f'Total matches  : {len(y)}')
print(f'Accuracy       : {accuracy:.1%}')
print(f'AUC-ROC        : {roc_auc_score(y, all_probs):.4f}')

results_df = df_raw[['team1', 'team2', 'venue']].copy()
results_df['actual_winner']    = y
results_df['p_team1_wins']     = all_probs.round(3)
results_df['predicted_winner'] = preds
results_df['correct']          = (preds == y)
results_df.head(10)

In [ ]:
# ROC curve + Precision-Recall curve
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Model Evaluation on Full Dataset (3000 matches)', fontsize=13, fontweight='bold')

fpr, tpr, _ = roc_curve(y, all_probs)
auc = roc_auc_score(y, all_probs)
axes[0].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc:.3f}')
axes[0].plot([0,1],[0,1], 'k--', lw=1)
axes[0].set(xlabel='False Positive Rate', ylabel='True Positive Rate', title='ROC Curve')
axes[0].legend(fontsize=11)

prec, rec, _ = precision_recall_curve(y, all_probs)
axes[1].plot(rec, prec, color='coral', lw=2)
axes[1].set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curve')

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance from saved results image — also show top features as a bar chart
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Feature Importance and Training History', fontsize=13, fontweight='bold')

img1 = mpimg.imread('../results/feature_importance.png')
axes[0].imshow(img1)
axes[0].axis('off')
axes[0].set_title('Permutation Feature Importance')

img2 = mpimg.imread('../results/training_history.png')
axes[1].imshow(img2)
axes[1].axis('off')
axes[1].set_title('Training History')

plt.tight_layout()
plt.show()

In [ ]:
# High-confidence predictions analysis
# How accurate is the model when it's very sure vs when it's uncertain?

bins = [(0.5, 0.6, 'Low (50-60%)'), (0.6, 0.7, 'Medium (60-70%)'),
        (0.7, 0.8, 'High (70-80%)'), (0.8, 1.01, 'Very High (80%+)')]

print(f'{"Confidence Band":<22} {"Matches":>8} {"Accuracy":>10} {"% of Dataset":>14}')
print('-' * 58)

max_prob = np.maximum(all_probs, 1 - all_probs)

for lo, hi, label in bins:
    mask = (max_prob >= lo) & (max_prob < hi)
    n = mask.sum()
    if n == 0:
        continue
    acc = (preds[mask] == y[mask]).mean()
    pct = n / len(y) * 100
    print(f'{label:<22} {n:>8}    {acc:>8.1%}    {pct:>11.1f}%')

print(f'\nKey insight: accuracy rises sharply as model confidence increases')